# Unit 3: CNN 基础

## 学习目标
- 理解卷积运算的直观含义和数学原理
- 掌握 `nn.Conv2d` 的核心参数：kernel_size, stride, padding, dilation
- 理解池化层的作用和类型
- 学会计算输出尺寸和感受野
- 搭建第一个 CNN 并在 MNIST 上训练

## 3.1 为什么需要 CNN？

全连接网络 (MLP) 处理图像的局限性：
1. **参数爆炸**：28x28 图像 -> 784 输入，第一层 256 神经元 = 784 × 256 = 200K 参数
2. **丢失空间结构**：图像被展平为一维向量，邻域关系被破坏
3. **无平移不变性**：物体在图像中移动，MLP 需要重新学习

CNN 的解决方案：
- **局部连接**：每个神经元只连接局部区域（感受野）
- **权重共享**：同一个卷积核在整张图上滑动，参数量大幅减少
- **层次化特征**：浅层检测边缘/纹理，深层检测语义/物体

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 3.2 卷积运算的直观理解

卷积核 (Kernel/Filter) 在输入图像上**滑动**，每次计算**逐元素乘积之和**。

```
输入 (5x5)          卷积核 (3x3)        输出 (3x3)
1  1  1  0  0       1  0  1            4  3  4
0  1  1  1  0       0  1  0            2  4  3
0  0  1  1  1       1  0  1            2  3  4
0  0  1  1  0    *              =
0  1  1  0  0

输出[0,0] = 1*1+1*0+1*1 + 0*0+1*1+1*0 + 0*1+0*0+1*1 = 4
```

In [ ]:
input_data = torch.tensor([[
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0]
]], dtype=torch.float32).unsqueeze(0)



kernel = torch.tensor([[
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
]], dtype=torch.float32).unsqueeze(0)

print(f"Input shape: {input_data.shape}")
print(f"Kernel shape: {kernel.shape}")

output = F.conv2d(input_data, kernel)
print(f"Output shape: {output.shape}")
print(f"Output:\n{output.squeeze()}")

### 边缘检测示例

索贝尔 sobel：一种常用的边缘检测算子，用于检测图像中的边缘。

不同的卷积核可以检测不同的图像特征。

In [ ]:
from torchvision.io import read_image
from torchvision.transforms.functional import resize

import urllib.request
from io import BytesIO
from PIL import Image

url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
try:
    with urllib.request.urlopen(url) as resp:
        img = Image.open(BytesIO(resp.read())).convert("L") # 转换为灰度图
    img_tensor = transforms.ToTensor()(img).unsqueeze(0)
    print(f"Image shape: {img_tensor.shape}")
    # transforms.ToTensor() 完成了从 [0, 255] 到 [0.0, 1.0] 的缩放
    print(f"Image values: {img_tensor[0, 0, :2, :2]}")


    # 检测垂直边缘（水平方向梯度 Gx）	左右两侧像素差值大 → 响应强
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
    # 检测水平边缘（垂直方向梯度 Gy）	上下两侧像素差值大 → 响应强
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
    sobel_x = sobel_x.view(1, 1, 3, 3)
    sobel_y = sobel_y.view(1, 1, 3, 3)

    edge_x = F.conv2d(img_tensor, sobel_x, padding=1)
    edge_y = F.conv2d(img_tensor, sobel_y, padding=1)
    # 数学本质是计算梯度向量的欧几里得范数（L2 模）
    edge_magnitude = torch.sqrt(edge_x ** 2 + edge_y ** 2)

    print(edge_x[0, 0, :2, :2])
    print(edge_y[0, 0, :2, :2])
    print(edge_magnitude[0, 0, :2, :2])

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    im0 = axes[0].imshow(img_tensor.squeeze(), cmap="gray", vmin=0, vmax=1)
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
    axes[0].set_title("Original")
    im1 = axes[1].imshow(edge_x.squeeze(), cmap="gray", vmin=-1, vmax=1)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    axes[1].set_title("Sobel X (vertical edges)")
    im2 = axes[2].imshow(edge_y.squeeze(), cmap="gray", vmin=-1, vmax=1)
    plt.colorbar(im2, ax=axes[2], fraction=0.046)
    axes[2].set_title("Sobel Y (horizontal edges)")
    im3 = axes[3].imshow(edge_magnitude.squeeze(), cmap="gray", vmin=0, vmax=1)
    plt.colorbar(im3, ax=axes[3], fraction=0.046)   
    axes[3].set_title("Sobel Magnitude (edge detection)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not load image: {e}")
    print("Skipping edge detection demo - this does not affect learning.")

## 3.3 nn.Conv2d 详解

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)
```

| 参数 | 含义 |
|------|------|
| `in_channels` | 输入通道数（RGB 图像 = 3） |
| `out_channels` | 输出通道数（=卷积核数量） |
| `kernel_size` | 卷积核尺寸，int 或 tuple |
| `stride` | 卷积核滑动步长，默认 1 |
| `padding` | 边缘填充，默认 0 |
| `dilation` | 空洞卷积率，默认 1 |

### 输出尺寸公式

$$H_{out} = \left\lfloor \frac{H_{in} + 2 \times padding - dilation \times (kernel\_size - 1) - 1}{stride} \right\rfloor  + 1 $$

简化版 (dilation=1)：
$$H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1 $$

out_channels=16 意味着卷积层内部有 16 组独立的卷积核，每组卷积核的形状为 (3, 3, 3)（即 in_channels × k × k）。  
每一组对整个输入的3个通道做完整卷积后产生 1 个特征图，16组就产生16个特征图，所以输出通道数 = out_channels = 16。

In [ ]:
x = torch.randn(1, 3, 32, 32)

conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
out = conv1(x)
print(f"Input:  {x.shape}  (N, C, H, W)")
print(f"Conv:   in=3, out=16, k=3, s=1, p=1")
print(f"Output: {out.shape}")
print(f"Expected: (1, 16, 32, 32)  # same size with padding=1")

In [ ]:
x = torch.randn(1, 3, 32, 32)

configs = [
    ("k=3, s=1, p=0", 3, 1, 0),
    ("k=3, s=1, p=1", 3, 1, 1),
    ("k=3, s=2, p=1", 3, 2, 1),
    ("k=5, s=1, p=2", 5, 1, 2),
    ("k=5, s=2, p=2", 5, 2, 2),
]

print(f"Input: {x.shape}, 计算: H_out = floor((32 + 2P - K) / S + 1)")
print("-" * 55)
for name, k, s, p in configs:
    conv = nn.Conv2d(3, 8, k, stride=s, padding=p)
    out = conv(x)
    expected_h = (32 + 2 * p - k) // s + 1
    print(f"{name:20s} -> Output: {str(out.shape):20s} (Expected H={expected_h})")

### 参数量计算

对于 `Conv2d(in, out, k)`：
$$\text{Params} = out \times in \times k \times k + out \quad (\text{+bias})$$

In [ ]:
for in_c, out_c, k in [(3, 16, 3), (16, 32, 3), (32, 64, 3)]:
    conv = nn.Conv2d(in_c, out_c, k)
    params = sum(p.numel() for p in conv.parameters())
    manual = out_c * in_c * k * k + out_c
    print(f"Conv2d({in_c}, {out_c}, {k}): {params:,} params (manual: {manual:,})")
    for name, param in conv.named_parameters():
        print(f"  {name}: {param.shape}")

## 3.4 池化层 (Pooling)

池化层**降采样**特征图，减少计算量，增强平移不变性。

| 类型 | 操作 | 适用场景 |
|------|------|---------|
| **MaxPool2d** | 取区域最大值 | 最常用，保留显著特征 |
| **AvgPool2d** | 取区域平均值 | 全局平均池化常用于分类头 |
| **AdaptiveAvgPool2d** | 自适应到指定输出尺寸 | 接受任意尺寸输入 |

In [ ]:
x = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
print(f"Input (4x4):\n{x.squeeze()}")

max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

print(f"\nMaxPool (2x2, stride=2):\n{max_pool(x).squeeze()}")
print(f"\nAvgPool (2x2, stride=2):\n{avg_pool(x).squeeze()}")

nn.AdaptiveAvgPool2d 是 PyTorch 中一个非常实用的池化层。  
与传统的 AvgPool2d（需要手动指定 kernel_size、stride、padding）不同，Adaptive（自适应）池化的核心思想是：你只需要指定输出尺寸，框架会自动计算所需的窗口大小和步长。

In [ ]:
x = torch.randn(4, 64, 7, 7)
print(f"Input: {x.shape}")

gap = nn.AdaptiveAvgPool2d((1, 1))
out = gap(x)
print(f"Global Avg Pool -> {out.shape}")

adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
out2 = adaptive_pool(x)
print(f"Adaptive (4,4) -> {out2.shape}")

## 3.5 感受野 (Receptive Field)

感受野 = 输出特征图上一个像素对应输入图像的区域大小。

递推公式（从前往后计算）：
$$RF_{i} = RF_{i-1} + (kernel\_size_i - 1) \times \prod_{j=1}^{i} stride_j$$
$$RF_0 = 1$$

更深层的神经元能看到更大的输入区域——这是 CNN 层级特征提取的关键。

In [ ]:
layers = [
    ("Conv1", "k3s1", 3, 1),
    ("Pool1", "k2s2", 2, 2),
    ("Conv2", "k3s1", 3, 1),
    ("Pool2", "k2s2", 2, 2),
    ("Conv3", "k3s1", 3, 1),
]

rf = 1 # 初始感受野为1（输入层的每个像素）
stride_prod = 1 # 累积步长（之前所有层步长的乘积）
print(f"{'Layer':<10s} {'Kernel':<8s} {'Stride':<8s} {'RF':<8s}")
print("-" * 35)
print(f"{'Conv0':<10s} {1:<8d} {1:<8d} {1:<8d}")
print("-" * 35)
for name, _, k, s in layers:
    rf = rf + (k - 1) * stride_prod
    stride_prod *= s # 累积当前层的步长
    print(f"{name:<10s} {k:<8d} {s:<8d} {rf:<8d}")

## 3.6 图像张量的维度规范

PyTorch 使用 **NCHW** 格式：
- **N**: Batch size (批大小)
- **C**: Channels (通道数，RGB=3, 灰度=1)
- **H**: Height (高度)
- **W**: Width (宽度)

每个维度上进行的操作：
- **Conv2d**: 在 H, W 上滑动，跨 C 求和
- **Pool2d**: 在 H, W 上降采样，对每个 C 独立
- **BatchNorm2d**: 在 N, H, W 上统计，对每个 C 独立

## 3.7 实战：构建第一个 CNN - LeNet 风格

经典架构：`Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> FC -> FC`

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # x.shape = (1, 16, 28, 28) -> (1, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x))) # x.shape = (1, 32, 14, 14) -> (1, 32, 7, 7)
        x = x.view(x.size(0), -1) # x.shape = (1, 32 * 7 * 7)
        x = F.relu(self.fc1(x)) # x.shape = (1, 128)
        x = self.dropout(x)
        x = self.fc2(x) # x.shape = (1, 10)
        return x

model = SimpleCNN()
print(model)

x = torch.randn(1, 1, 28, 28)
with torch.no_grad():
    out = model(x)
print(f"\nInput {x.shape} -> Output {out.shape}")
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

### 特征图尺寸追踪

In [ ]:
x = torch.randn(1, 1, 28, 28)
print(f"Input:              {x.shape}")

x1 = F.relu(model.conv1(x))
print(f"After conv1+ReLU:   {x1.shape}  (28x28, 16 channels)")

x2 = model.pool(x1)
print(f"After pool1:        {x2.shape}  (14x14, 16 channels)")

x3 = F.relu(model.conv2(x2))
print(f"After conv2+ReLU:   {x3.shape}  (14x14, 32 channels)")

x4 = model.pool(x3)
print(f"After pool2:        {x4.shape}  (7x7, 32 channels)")

x5 = x4.view(x4.size(0), -1)
print(f"After flatten:      {x5.shape}  (32*7*7 = {32*7*7})")

x6 = F.relu(model.fc1(x5))
print(f"After fc1+ReLU:     {x6.shape}  (128 channels)")

x7 = model.dropout(x6)
print(f"After dropout:      {x7.shape}  (128 channels)")

x8 = model.fc2(x7)
print(f"After fc2:          {x8.shape}  (10 channels)")

### 训练 SimpleCNN 在 MNIST 上

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total

In [ ]:
model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

epochs = 8
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], "b-", label="Train")
ax1.plot(history["test_loss"], "r-", label="Test")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], "b-", label="Train")
ax2.plot(history["test_acc"], "r-", label="Test")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f"SimpleCNN on MNIST (Final Test Acc: {history['test_acc'][-1]:.2%})", fontsize=13)
plt.tight_layout()
plt.show()

## 3.8 CNN vs MLP 对比

| 特征 | MLP (Unit 2) | SimpleCNN (Unit 3) |
|------|-------------|-------------------|
| 参数量 | ~240K | ~55K |
| 准确率 (MNIST) | ~97-98% | ~98-99% |
| 空间结构保留 | 否 (展平) | 是 |
| 平移不变性 | 无 | 有 (池化) |

CNN 用**更少的参数**达到了**更高的准确率**！

## 3.9 单元小结

| 概念 | 要点 |
|------|------|
| **卷积** | 局部连接 + 权重共享，在 H,W 上滑动 |
| **Conv2d 参数** | in_channels, out_channels, kernel_size, stride, padding |
| **输出尺寸** | $H_{out} = \lfloor(H + 2P - K)/S + 1\rfloor$ |
| **池化** | MaxPool2d (保留显著) / AvgPool2d (平滑) |
| **NCHW** | Batch, Channel, Height, Width - 标准维度顺序 |
| **感受野** | 深层神经元对应更大的输入区域 |

### 思考题
1. 为什么 padding=1, kernel_size=3 可以保持尺寸不变？
2. 如果 stride=2，输出尺寸是多少？（设 H=32, K=3, P=1）
3. 为什么 CNN 比 MLP 参数更少但效果更好？